# 02 - Exploratory Data Analysis

Stage 2 of 4. Reads the artifacts written by `01_data_loading.ipynb` and reproduces
every exploration table and plot from the original project. Each figure is saved
into `output/plots/`.

**What the plots are for:** the data is heavily imbalanced, and fraud looks different
from normal spend - that is the premise the whole thesis rests on.

## 1. Imports & setup

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (8, 5)
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

In [ ]:
from pathlib import Path

KAGGLE_IN = Path("/kaggle/input")


def find_data_dir():
    """Folder holding fraudTrain.csv / fraudTest.csv (Kaggle input mount or local archive/)."""
    for c in [KAGGLE_IN / "fraud-detection", Path("archive"), Path(".")]:
        if (c / "fraudTrain.csv").exists():
            return c
    for p in sorted(KAGGLE_IN.rglob("fraudTrain.csv")) if KAGGLE_IN.exists() else []:
        return p.parent
    return Path("archive")


# Raw CSVs: Kaggle dataset mount when on Kaggle, else the local archive/ folder.
DATA_DIR = find_data_dir()

# Everything this project generates goes into output/.
OUT = Path("/kaggle/working/output") if Path("/kaggle/working").exists() else Path("output")
for sub in ["data", "plots", "models", "preds", "results"]:
    (OUT / sub).mkdir(parents=True, exist_ok=True)


def upstream(rel):
    """Locate a file written by an earlier notebook (local run or Kaggle kernel input)."""
    local = OUT / rel
    if local.exists():
        return local
    if KAGGLE_IN.exists():
        for p in sorted(KAGGLE_IN.rglob(rel.split("/")[-1])):
            if p.as_posix().endswith("output/" + rel):
                return p
    raise FileNotFoundError("Run the earlier notebook first - missing: " + rel)


def save_fig(name):
    """Save the current matplotlib figure into output/plots/."""
    plt.savefig(OUT / "plots" / (name + ".png"), dpi=150, bbox_inches="tight")


print("DATA_DIR:", DATA_DIR)
print("OUT     :", OUT)

## 2. Load the prepared data from notebook 01

In [ ]:
train_raw = pd.read_parquet(upstream("data/train_eda.parquet"))
train_fe = pd.read_parquet(upstream("data/train_fe.parquet"))

print("Raw EDA columns :", train_raw.shape)
print("Engineered train:", train_fe.shape)
train_raw.head(3)

## 3. Class balance

In [ ]:
plt.figure(figsize=(6, 4))

sns.countplot(
    x="is_fraud",
    data=train_raw
)

plt.title("Fraud Distribution")
save_fig("01_fraud_distribution")
plt.show()

print(train_raw["is_fraud"].value_counts())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

counts = train_raw["is_fraud"].value_counts().sort_index()
axes[0].bar(["Not Fraud (0)", "Fraud (1)"], counts.values, color=["#4C78A8", "#E45756"])
axes[0].set_title("Class imbalance (Train)")
axes[0].set_ylabel("Number of transactions")
for i, v in enumerate(counts.values):
    axes[0].text(i, v, f"{v:,}", ha="center", va="bottom", fontsize=10)

axes[1].pie(
    counts.values,
    labels=["Not Fraud", "Fraud"],
    autopct="%1.2f%%",
    colors=["#4C78A8", "#E45756"],
    startangle=90,
    explode=(0, 0.15),
)
axes[1].set_title("Fraud share (Train)")
plt.tight_layout()
save_fig("02_class_imbalance")
plt.show()

print("Imbalance ratio (non-fraud : fraud) = {:.1f} : 1".format(counts[0] / counts[1]))

In [ ]:
fraud_rate = train_raw['is_fraud'].mean()*100
print(fraud_rate)

## 4. Transaction amount

In [ ]:
train_raw.groupby("is_fraud")["amt"].describe()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.boxplot(data=train_raw, x="is_fraud", y="amt", ax=axes[0], palette=["#4C78A8", "#E45756"])
axes[0].set_title("Transaction amount by class")
axes[0].set_xticklabels(["Not Fraud", "Fraud"])
axes[0].set_ylabel("Amount")

# Cap y for readability (outliers still exist but plot stays readable)
q99 = train_raw["amt"].quantile(0.99)
sns.histplot(
    data=train_raw[train_raw["amt"] <= q99],
    x="amt",
    hue="is_fraud",
    bins=40,
    element="step",
    stat="density",
    common_norm=False,
    ax=axes[1],
    palette=["#4C78A8", "#E45756"],
)
axes[1].set_title("Amount distribution (up to 99th percentile)")
plt.tight_layout()
save_fig("03_amount_by_class")
plt.show()

print(train_raw.groupby("is_fraud")["amt"].describe()[["mean", "50%", "std"]])

## 5. Time of day

In [ ]:
tmp = train_raw.copy()
tmp["trans_date_trans_time"] = pd.to_datetime(tmp["trans_date_trans_time"])
tmp["hour"] = tmp["trans_date_trans_time"].dt.hour

hourly = tmp.groupby("hour")["is_fraud"].mean() * 100

plt.figure(figsize=(10, 4))
plt.plot(hourly.index, hourly.values, marker="o", color="#E45756")
plt.title("Fraud rate (%) by hour of day")
plt.xlabel("Hour")
plt.ylabel("Fraud rate (%)")
plt.xticks(range(0, 24))
plt.tight_layout()
save_fig("04_fraud_rate_by_hour")
plt.show()

## 6. Merchant category

In [ ]:
fraud_rate = (
    train_raw.groupby("category")["is_fraud"]
      .agg(["count", "sum", "mean"])
      .sort_values("mean", ascending=False)
)

fraud_rate

In [ ]:
cat_fraud = (
    train_raw.groupby("category")["is_fraud"]
    .mean()
    .mul(100)
    .sort_values(ascending=False)
)

plt.figure(figsize=(10, 5))
sns.barplot(x=cat_fraud.values, y=cat_fraud.index, color="#E45756")
plt.title("Fraud rate (%) by merchant category")
plt.xlabel("Fraud rate (%)")
plt.ylabel("Category")
plt.tight_layout()
save_fig("05_fraud_rate_by_category")
plt.show()

## 7. Engineered features vs fraud

In [ ]:
# Quick check: engineered features vs fraud
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.boxplot(data=train_fe, x="is_fraud", y="distance_km", ax=axes[0], palette=["#4C78A8", "#E45756"])
axes[0].set_title("Customer-merchant distance by class")
axes[0].set_xticklabels(["Not Fraud", "Fraud"])

night_rate = train_fe.groupby("is_night")["is_fraud"].mean() * 100
axes[1].bar(["Day (6-23)", "Night (0-5)"], night_rate.values, color=["#4C78A8", "#E45756"])
axes[1].set_title("Fraud rate (%) by night flag")
axes[1].set_ylabel("Fraud rate (%)")
for i, v in enumerate(night_rate.values):
    axes[1].text(i, v, f"{v:.2f}%", ha="center", va="bottom")

plt.tight_layout()
save_fig("06_engineered_features")
plt.show()

## 8. Save the EDA tables

In [ ]:
train_raw.groupby("is_fraud")["amt"].describe().to_csv(OUT / "results" / "eda_amount_by_class.csv")
fraud_rate.to_csv(OUT / "results" / "eda_fraud_rate_by_category.csv")
hourly.rename("fraud_rate_pct").to_csv(OUT / "results" / "eda_fraud_rate_by_hour.csv")

print("Saved plots:")
for p in sorted((OUT / "plots").iterdir()):
    print(" ", p.name)
print("\nSaved tables:")
for p in sorted((OUT / "results").iterdir()):
    print(" ", p.name)